# PBIX Documenter -- Core Pattern Walkthrough

This notebook demonstrates the core engineering patterns in PBIX Documenter without requiring a real `.pbix` file.

It covers:
1. What `pbixray` extracts from a PBIX file (illustrated with synthetic data)
2. The `FieldDefinitionCatalog` enrichment pattern
3. The Markdown-first generation approach
4. The LLM explanation pattern (illustrated with a mock LLM)
5. The multi-format export pipeline

---

To run against a real `.pbix` file, replace the synthetic DataFrames in Section 1 with:
```python
from pbixray import PBIXRay
model = PBIXRay('your_file.pbix')
```

## 1. Synthetic PBIX Model Data

`pbixray` parses a `.pbix` file and exposes its semantic model as pandas DataFrames.
Here we construct equivalent synthetic DataFrames to illustrate the structure.

In [ ]:
import pandas as pd

# Tables in the model
tables = ['Sales', 'Customer', 'Date']

# Schema: all columns across all tables
schema = pd.DataFrame([
    {'TableName': 'Sales',    'ColumnName': 'OrderDate',     'PandasDataType': 'datetime64'},
    {'TableName': 'Sales',    'ColumnName': 'OrderAmount',   'PandasDataType': 'float64'},
    {'TableName': 'Sales',    'ColumnName': 'CustomerID',    'PandasDataType': 'int64'},
    {'TableName': 'Sales',    'ColumnName': 'ProductCategory','PandasDataType': 'object'},
    {'TableName': 'Customer', 'ColumnName': 'CustomerID',    'PandasDataType': 'int64'},
    {'TableName': 'Customer', 'ColumnName': 'CustomerName',  'PandasDataType': 'object'},
    {'TableName': 'Customer', 'ColumnName': 'RegionName',    'PandasDataType': 'object'},
    {'TableName': 'Date',     'ColumnName': 'Date',          'PandasDataType': 'datetime64'},
    {'TableName': 'Date',     'ColumnName': 'FiscalYear',    'PandasDataType': 'int64'},
    {'TableName': 'Date',     'ColumnName': 'FiscalQuarter', 'PandasDataType': 'object'},
])

# DAX measures
dax_measures = pd.DataFrame([
    {'TableName': 'Sales', 'Name': 'Total Revenue',
     'Expression': 'SUM(Sales[OrderAmount])',
     'Description': 'Sum of all order amounts'},
    {'TableName': 'Sales', 'Name': 'Customer Count',
     'Expression': 'DISTINCTCOUNT(Sales[CustomerID])',
     'Description': 'Count of unique customers with orders'},
    {'TableName': 'Sales', 'Name': 'Revenue YoY%',
     'Expression': 
     'DIVIDE([Total Revenue] - CALCULATE([Total Revenue], SAMEPERIODLASTYEAR(Date[Date])), '
     'CALCULATE([Total Revenue], SAMEPERIODLASTYEAR(Date[Date])))',
     'Description': 'Year-over-year revenue growth percentage'},
])

# DAX calculated columns
dax_columns = pd.DataFrame([
    {'TableName': 'Customer', 'ColumnName': 'FullLabel',
     'Expression': 'Customer[CustomerName] & " (" & Customer[RegionName] & ")"'},
])

# Relationships between tables
relationships = pd.DataFrame([
    {'FromTableName': 'Sales', 'FromColumnName': 'CustomerID',
     'ToTableName': 'Customer', 'ToColumnName': 'CustomerID',
     'Cardinality': 'ManyToOne', 'CrossFilteringBehavior': 'OneDirection',
     'IsActive': True, 'RelyOnReferentialIntegrity': False},
    {'FromTableName': 'Sales', 'FromColumnName': 'OrderDate',
     'ToTableName': 'Date', 'ToColumnName': 'Date',
     'Cardinality': 'ManyToOne', 'CrossFilteringBehavior': 'OneDirection',
     'IsActive': True, 'RelyOnReferentialIntegrity': False},
])

# Power Query (M) scripts
power_query = pd.DataFrame([
    {'TableName': 'Sales',
     'Expression': 'let\n    Source = Sql.Database("server", "db"),\n    Sales = Source{[Name="Sales"]}[Data]\nin\n    Sales'},
    {'TableName': 'Customer',
     'Expression': 'let\n    Source = Sql.Database("server", "db"),\n    Customer = Source{[Name="Customer"]}[Data]\nin\n    Customer'},
])

print('Model components loaded:')
print(f'  Tables: {len(tables)}')
print(f'  Schema columns: {len(schema)}')
print(f'  DAX measures: {len(dax_measures)}')
print(f'  DAX calculated columns: {len(dax_columns)}')
print(f'  Relationships: {len(relationships)}')
print(f'  Power Query scripts: {len(power_query)}')

## 2. Field Definition Catalog Enrichment

The catalog maps column names to business-friendly definitions.
It is loaded once per generation run and the lookup is memoized.

In [ ]:
import re
from functools import lru_cache

# Load the sample catalog
catalog = pd.read_csv('../sample_catalog.csv')
print(f'Catalog loaded: {len(catalog)} field definitions')
catalog.head()

In [ ]:
def build_lookup(catalog_df, primary_key, fallback_key, definition_cols):
    """Build a memoized lookup function from a catalog DataFrame."""

    def normalize(text):
        return re.sub(r'\s+', ' ', str(text)).strip()

    @lru_cache(maxsize=None)
    def lookup(field_name):
        row = catalog_df[catalog_df[primary_key].str.lower() == field_name.lower()]
        if row.empty:
            row = catalog_df[catalog_df[fallback_key].str.lower() == field_name.lower()]
        if row.empty:
            return ''
        for col in definition_cols:
            val = row.iloc[0][col]
            if pd.notna(val) and str(val).strip().lower() not in ('not applicable', ''):
                return normalize(val)
        return ''

    return lookup

get_definition = build_lookup(
    catalog,
    primary_key='pbi_column_name',
    fallback_key='source_column_name',
    definition_cols=['business_definition', 'technical_definition']
)

# Test the lookup
test_fields = ['CustomerID', 'OrderAmount', 'RegionName', 'UnknownField']
for f in test_fields:
    print(f'{f!r:20} -> {get_definition(f)!r}')

In [ ]:
# Enrich the schema DataFrame with definitions upfront
enriched_schema = schema.copy()
enriched_schema['Definition'] = enriched_schema['ColumnName'].apply(get_definition)
enriched_schema.fillna('', inplace=True)

print('Enriched schema (Sales table):')
enriched_schema[enriched_schema['TableName'] == 'Sales']

## 3. Markdown-First Generation

The generator iterates over the model and appends Markdown lines to a list.
The final document is produced by joining all lines with newlines.

This is the core of the Markdown-first design: one canonical string,
no format-specific logic in the generation path.

In [ ]:
def generate_markdown(tables, enriched_schema, dax_measures, dax_columns,
                      power_query, relationships, explain_fn=None):
    """
    Generate a canonical Markdown document from model component DataFrames.

    explain_fn: optional callable(code, label) -> str for LLM explanations.
    """
    if explain_fn is None:
        explain_fn = lambda code, label: ''  # noqa: E731

    lines = [
        '# Sample Model - Model Documentation',
        '**File**: `sample_model.pbix`',
        '',
    ]

    for table in tables:
        lines.append(f'## Table: {table}')

        # M Query
        if not power_query.empty:
            pq = power_query[power_query['TableName'] == table]
            for _, r in pq.iterrows():
                expr = str(r.get('Expression', '')).strip()
                if expr:
                    lines += ['#### M Query', '```m', expr, '```']
                    explanation = explain_fn(expr, f'M Query for {table}')
                    if explanation:
                        lines.append(explanation)

        # Relationships
        if not relationships.empty:
            rels = relationships[
                (relationships['FromTableName'] == table) |
                (relationships['ToTableName'] == table)
            ]
            if not rels.empty:
                lines += [
                    '', '#### Relationships', '',
                    '| Relationship | Cardinality | Cross Filter Direction | Is Active |',
                    '|---|---|---|---|',
                ]
                for _, r in rels.iterrows():
                    rel = f"`{r['FromTableName']}.[{r['FromColumnName']}] -> {r['ToTableName']}.[{r['ToColumnName']}]`"
                    lines.append(f"| {rel} | {r['Cardinality']} | {r['CrossFilteringBehavior']} | {'Yes' if r['IsActive'] else 'No'} |")
                lines.append('')

        # Schema fields
        s = enriched_schema[enriched_schema['TableName'] == table]
        if not s.empty:
            lines += ['', '#### Fields', '', '| Name | Type | Definition |', '|---|---|---|']
            for _, r in s.iterrows():
                lines.append(f"| {r['ColumnName']} | {r['PandasDataType']} | {r['Definition']} |")

        # Calculated columns
        if not dax_columns.empty:
            cols = dax_columns[dax_columns['TableName'] == table]
            if not cols.empty:
                lines.append('#### Calculated Columns')
            for _, r in cols.iterrows():
                expr = str(r.get('Expression', '')).strip()
                if expr:
                    lines += [f"**{r['ColumnName']}**", '```dax', expr, '```']
                    explanation = explain_fn(expr, f"column {r['ColumnName']}")
                    if explanation:
                        lines.append(explanation)
                    lines.append('')

        # DAX Measures
        if not dax_measures.empty:
            ms = dax_measures[dax_measures['TableName'] == table]
            if not ms.empty:
                lines.append('#### DAX Measures')
            for _, r in ms.iterrows():
                expr = str(r.get('Expression', '')).strip()
                if expr:
                    lines += [f"**{r['Name']}**", '```dax', expr, '```']
                    if r.get('Description'):
                        lines.append(f"Description: {r['Description']}")
                    explanation = explain_fn(expr, f"measure {r['Name']}")
                    if explanation:
                        lines.append(explanation)
                    lines.append('')

        lines.append('')

    return '\n'.join(lines)

mdata = generate_markdown(tables, enriched_schema, dax_measures, dax_columns, power_query, relationships)
print(mdata[:1500])
print('...')

## 4. LLM Explanation Pattern (Mock)

In production, `explain_fn` calls Ollama via LangChain.
Here we use a mock that returns a static string to demonstrate
the interface without requiring Ollama to be running.

In [ ]:
# Mock LLM explanation function
def mock_explain(code, label):
    """Simulates LLM output for demonstration purposes."""
    templates = {
        'SUM': 'Calculates the total sum of the specified column across the current filter context.',
        'DISTINCTCOUNT': 'Counts the number of unique values in the specified column.',
        'DIVIDE': 'Performs safe division, returning blank instead of an error when the denominator is zero.',
        'SAMEPERIODLASTYEAR': 'Returns a table of dates shifted one year back for year-over-year comparisons.',
    }
    for keyword, explanation in templates.items():
        if keyword in code:
            return f'_{explanation}_'
    return '_Performs a calculation on the data model using DAX._'

# Generate with mock explanations
mdata_with_explanations = generate_markdown(
    tables, enriched_schema, dax_measures, dax_columns, power_query, relationships,
    explain_fn=mock_explain
)

# Show just the Sales table section
sales_section_start = mdata_with_explanations.index('## Table: Sales')
sales_section_end = mdata_with_explanations.index('## Table: Customer')
print(mdata_with_explanations[sales_section_start:sales_section_end])

## 5. Multi-Format Export Pipeline

The Markdown string is the single source of truth.
The exporter converts it to HTML (and from HTML to Word/PDF).
Here we demonstrate the HTML conversion locally.

In [ ]:
import markdown2
import re

def markdown_to_html(mdata):
    """Convert canonical Markdown to styled HTML."""

    def escape_underscores(text):
        return text.replace('_', r'\_')

    mdata = mdata.replace('```dax', '```').replace('```m', '```')
    segments = re.split(r'(```.*?```)', mdata, flags=re.DOTALL)
    processed = []
    for segment in segments:
        if segment.startswith('```'):
            processed.append(segment)
        else:
            sub_segments = re.split(r'(`.*?`)', segment, flags=re.DOTALL)
            processed.append(''.join(
                s if s.startswith('`') else escape_underscores(s)
                for s in sub_segments
            ))

    html = markdown2.markdown(''.join(processed), extras=['tables'])
    html = re.sub(r'<code>\n', '<code>', html)
    return html

html_output = markdown_to_html(mdata_with_explanations)

# Save to a local file for inspection
with open('sample_output.html', 'w', encoding='utf-8') as f:
    f.write(f'<html><body>{html_output}</body></html>')

print('HTML output written to sample_output.html')
print(f'HTML length: {len(html_output)} characters')
print()
print('First 500 chars of HTML:')
print(html_output[:500])

## Summary

This notebook demonstrated the four core patterns in PBIX Documenter:

| Pattern | Key insight |
|---|---|
| PBIX extraction | `pbixray` exposes the semantic model as typed DataFrames |
| Catalog enrichment | Memoized lookup with graceful fallback keeps generation clean |
| Markdown-first generation | One canonical string, all formats derived from it |
| Export pipeline | HTML conversion is the only shared step; Word and PDF are trivial from there |

The production Streamlit app adds real-time streaming and progress feedback
via the callback pattern described in [ADR-003](../docs/adr/ADR-003-callback-streaming.md).